# vMAR — G0.9-P.1 · Interoperabilidad Python–C23

Laboratorio experimental, sin redes visuales ni entrenamiento independiente. Python invoca directamente la implementación C23 de Euler y RK4. **Primero ejecutar todas las celdas en orden.** Requiere un entorno Linux Colab con CMake, Git y compilador C23.

Esta prueba **no** valida aún la totalidad de memoria, control, redes multicapa ni paridad con FormalSpec.

In [ ]:
import os, subprocess, sys
subprocess.run(['git', '--version'], check=True)
subprocess.run(['cmake', '--version'], check=True)
subprocess.run(['cc', '--version'], check=True)

In [ ]:
from pathlib import Path
root = Path('/content/vMAR')
if root.exists():
    raise RuntimeError('Existe /content/vMAR; utilice un entorno limpio para reproducir.')
USE_PUBLIC_GITHUB = False  # Cambiar a True una vez publicado G0.9-P.1
if USE_PUBLIC_GITHUB:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RubenMoreValencia/vMAR.git', str(root)], check=True)
    revision = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
    print('Revisión exacta del código:', revision)
else:
    from google.colab import files
    from zipfile import ZipFile
    print('Seleccione vMAR_G0.9-P1_Colab_Experimental.zip descargado de esta conversación')
    uploaded = files.upload()
    archive_name = 'vMAR_G0.9-P1_Colab_Experimental.zip'
    assert archive_name in uploaded, 'No se cargó el paquete esperado'
    root.mkdir()
    with ZipFile(archive_name) as archive:
        archive.extractall(root)
    revision = 'paquete experimental G0.9-P.1; sin SHA público'
    print('Origen:', revision)
assert (root / 'python' / 'vmar_ctypes.py').is_file(), 'Falta el adaptador Python G0.9-P.1'

In [ ]:
build = root / 'build-colab'
subprocess.run(['cmake', '-S', str(root), '-B', str(build), '-DCMAKE_BUILD_TYPE=Release', '-DVMAR_BUILD_TESTS=OFF', '-DVMAR_BUILD_EXPERIMENTAL_CURVATURE=OFF', '-DVMAR_BUILD_PYTHON_FFI=ON'], check=True)
subprocess.run(['cmake', '--build', str(build), '--target', 'vmar_ffi', '-j', '2'], check=True)
shared = build / 'libvmar_ffi.so'
assert shared.is_file(), f'No se generó {shared}'
os.environ['VMAR_FFI_LIBRARY'] = str(shared)
sys.path.insert(0, str(root / 'python'))
print('Biblioteca C23 compartida:', shared)

In [ ]:
import math
from vmar_ctypes import VmarDifferential
motor = VmarDifferential(os.environ['VMAR_FFI_LIBRARY'])
campo = lambda t, y: (-y[0],)
exacta = math.exp(-0.1)
euler = motor.step(campo, (1.0,), step=0.1, method='euler')[0]
rk4 = motor.step(campo, (1.0,), step=0.1, method='rk4')[0]
print('Exacta:', exacta, 'Euler:', euler, 'RK4:', rk4)
print('Error Euler:', abs(euler-exacta), 'Error RK4:', abs(rk4-exacta))
assert abs(euler - 0.9) < 1e-14
assert abs(rk4 - 0.9048375) < 1e-14
assert abs(rk4-exacta) < abs(euler-exacta)
print('G0.9-P.1 ANALYTIC GATE PASS')

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(root/'python'), '-p', 'test_interop.py', '-v'], check=True, env=os.environ.copy())
print('G0.9-P.1 PYTHON-C23 CONTRACT GATE PASS')

## Informe mínimo de reproducibilidad

Registrar: fecha, revisión Git mostrada, compilador, CMake, versión de Python, método, resultado, error absoluto y estado de las tres pruebas. El objetivo es **interoperabilidad numérica delimitada**, no superioridad universal de RK4 ni demostración completa de aprendizaje neuronal.